In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)

n = 1000

# -----------------------------
# Categorical columns
# -----------------------------

gender = np.random.choice(
    ["Male", "Female"],
    size=n,
    p=[0.55, 0.45]
)

city = np.random.choice(
    ["Mumbai", "Pune", "Delhi", "Bangalore", "Hyderabad", "Chennai"],
    size=n
)

education = np.random.choice(
    ["High School", "Bachelor", "Master", "PhD"],
    size=n,
    p=[0.20, 0.45, 0.30, 0.05]
)

# Education -> experience tendency
education_bonus = {
    "High School": 0,
    "Bachelor": 1,
    "Master": 2,
    "PhD": 3
}

# -----------------------------
# Numerical columns
# -----------------------------

age = np.random.randint(21, 51, size=n)

# Experience related to age
experience = np.maximum(
    0,
    age - np.random.randint(20, 26, size=n)
)

# Add some education influence
experience = experience + np.array(
    [education_bonus[e] for e in education]
)

experience = np.clip(experience, 0, 30)


# Job level depends mainly on experience
job_level = np.select(
    [
        experience < 3,
        experience < 7,
        experience < 12,
        experience >= 12
    ],
    [
        "Junior",
        "Mid",
        "Senior",
        "Manager"
    ],
    default="Junior"
)


# Department
department = np.random.choice(
    ["Engineering", "Marketing", "Finance", "HR", "Sales"],
    size=n,
    p=[0.30, 0.15, 0.15, 0.10, 0.30]
)


# Performance score
performance = np.clip(
    50
    + experience * 1.2
    + np.random.normal(0, 8, n),
    40,
    100
).round(1)


# Projects completed depends on experience
projects = np.maximum(
    0,
    (experience * 1.5 + np.random.normal(0, 3, n))
).round().astype(int)


# Salary depends on:
# experience + education + job level + performance

education_salary_bonus = np.array([
    education_bonus[e] for e in education
])

job_salary_bonus = np.select(
    [
        job_level == "Junior",
        job_level == "Mid",
        job_level == "Senior",
        job_level == "Manager"
    ],
    [
        0,
        15000,
        35000,
        60000
    ]
)

salary = (
    25000
    + experience * 3500
    + education_salary_bonus * 7000
    + job_salary_bonus
    + performance * 400
    + np.random.normal(0, 5000, n)
)

salary = np.maximum(salary, 20000).round(0)


# -----------------------------
# Boolean / binary columns
# -----------------------------

has_degree = np.isin(
    education,
    ["Bachelor", "Master", "PhD"]
)

is_employed = np.ones(n, dtype=bool)


# -----------------------------
# Final DataFrame
# -----------------------------

df = pd.DataFrame({
    "Age": age,
    "Gender": gender,
    "City": city,
    "Education_Level": education,
    "Department": department,
    "Experience_Years": experience,
    "Job_Level": job_level,
    "Performance_Score": performance,
    "Projects_Completed": projects,
    "Monthly_Salary": salary,
    "Has_Degree": has_degree,
    "Is_Employed": is_employed
})

df.sample(10)


,Age,Gender,City,Education_Level,Department,Experience_Years,Job_Level,Performance_Score,Projects_Completed,Monthly_Salary,Has_Degree,Is_Employed
936,22,Male,Pune,Master,Sales,4,Mid,64.5,4,93788.0,True,True
503,25,Female,Bangalore,Bachelor,Finance,2,Junior,54.9,1,58079.0,True,True
626,47,Male,Hyderabad,Bachelor,Engineering,23,Manager,84.8,34,220726.0,True,True
899,26,Male,Pune,High School,Engineering,6,Mid,58.2,6,76383.0,False,True
673,25,Male,Mumbai,Bachelor,Engineering,4,Mid,59.1,6,77572.0,True,True
931,29,Female,Delhi,Master,Engineering,7,Senior,63.5,8,123475.0,True,True
878,34,Male,Chennai,High School,Sales,11,Senior,46.9,18,117463.0,False,True
304,29,Female,Chennai,Bachelor,Marketing,9,Senior,59.8,13,108258.0,True,True
421,33,Female,Mumbai,High School,Engineering,8,Senior,61.8,10,107778.0,False,True
680,45,Female,Mumbai,Bachelor,Sales,26,Manager,89.0,38,216354.0,True,True


In [2]:
cols = ["Has_Degree", "Education_Level", "Job_Level"]
df2 = df[cols]
df2.sample(10)

,Has_Degree,Education_Level,Job_Level
359,False,High School,Manager
471,False,High School,Manager
655,True,PhD,Manager
997,True,Master,Junior
84,False,High School,Manager
665,True,Master,Mid
675,False,High School,Senior
456,False,High School,Manager
639,False,High School,Mid
296,True,Master,Manager


In [3]:
X = df2.iloc[: , 0:2]
y = df2.iloc[: , 2]
X.head()

,Has_Degree,Education_Level
0,True,Bachelor
1,True,Bachelor
2,True,Bachelor
3,True,Master
4,True,Bachelor


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [5]:
X_train.head()

,Has_Degree,Education_Level
533,True,Master
777,True,Bachelor
714,True,Bachelor
716,True,Bachelor
260,True,Bachelor


In [6]:
from sklearn.preprocessing import OrdinalEncoder

oe = OrdinalEncoder(categories=[[False, True], ["High School","Bachelor", "Master", "PhD"]])


In [7]:
oe.fit(X_train)

,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute.","[[False, True], ['High School', 'Bachelor', ...]]"
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"handle_unknown handle_unknown: {'error', 'use_encoded_value'}, default='error'When set to 'error' an error will be raised in case an unknowncategorical feature is present during transform. When set to'use_encoded_value', the encoded value of unknown categories will beset to the value given for the parameter `unknown_value`. In:meth:`inverse_transform`, an unknown category will be denoted as None... versionadded:: 0.24",'error'
,"unknown_value unknown_value: int or np.nan, default=NoneWhen the parameter handle_unknown is set to 'use_encoded_value', thisparameter is required and will set the encoded value of unknowncategories. It has to be distinct from the values used to encode any ofthe categories in `fit`. If set to np.nan, the `dtype` parameter mustbe a float dtype... versionadded:: 0.24",None
,"encoded_missing_value encoded_missing_value: int or np.nan, default=np.nanEncoded value of missing categories. If set to `np.nan`, then the `dtype`parameter must be a float dtype... versionadded:: 1.1",nan
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.3 Read more in the :ref:`User Guide <encoder_infrequent_categories>`.",None
,"max_categories max_categories: int, default=NoneSpecifies an upper limit to the number of output categories for each inputfeature when considering infrequent categories. If there are infrequentcategories, `max_categories` includes the category representing theinfrequent categories along with the frequent categories. If `None`,there is no limit to the number of output features.`max_categories` do **not** take into account missing or unknowncategories. Setting `unknown_value` or `encoded_missing_value` to aninteger will increase the number of unique integer codes by one each.This can result in up to `max_categories + 2` integer codes... versionadded:: 1.3 Read more in the :ref:`User Guide <encoder_infrequent_categories>`.",None
Name,Type,Value
categories_ categories_: list of arraysThe categories of each feature determined during ``fit`` (in order ofthe features in X and corresponding with the output of ``transform``).This does not include categories that weren't seen during ``fit``.,list,"[array([False, True]), array(['High ... dtype=object)]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](2,)","['Has_Degree','Education_Level']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 1.0,int,2


In [8]:
X_train = oe.transform(X_train)
X_test = oe.transform(X_test)

In [9]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
le.fit(y_test)

Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)Holds the label for each class.","ndarray[object](4,)","['Junior','Manager','Mid','Senior']"


In [10]:
le.classes_

array(['Junior', 'Manager', 'Mid', 'Senior'], dtype=object)

In [11]:
y_test = le.transform(y_test)
y_train = le.transform(y_train)

In [13]:
y_test

array([1, 2, 3, 1, 0, 0, 1, 0, 2, 1, 1, 1, 3, 1, 1, 1, 0, 1, 3, 1, 1, 2,
       1, 1, 0, 3, 1, 1, 2, 3, 1, 1, 2, 1, 1, 3, 1, 3, 3, 1, 0, 0, 1, 2,
       1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 2, 1, 0, 3, 1, 1, 3, 1, 1, 1,
       3, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 0, 1, 1, 1, 1, 2, 1, 3, 1, 1, 3,
       1, 0, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 0, 1, 1,
       1, 0, 1, 3, 1, 3, 1, 2, 1, 1, 2, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 2, 1, 1, 1, 3, 1, 1, 1, 2, 1, 2, 0, 1, 2, 1, 0, 1, 1, 1, 3, 3,
       0, 2, 1, 1, 2, 1, 1, 1, 2, 1, 1, 1, 1, 2, 1, 3, 3, 1, 1, 1, 1, 1,
       2, 2, 1, 3, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 0,
       1, 2])